In [ ]:
from pathlib import Path
import pandas as pd
from collections import Counter

PROJECT_ROOT = Path(".")
CSV_PATH = PROJECT_ROOT / "data" / "selected_30k.csv"

df = pd.read_csv(CSV_PATH)


def get_tile_from_patch_id(patch_id):
    return patch_id.rsplit("_", 2)[0]


df["tile"] = df["patch_id"].apply(get_tile_from_patch_id)

tile_counts = (
    df["tile"]
    .value_counts()
    .sort_values(ascending=False)
)

print("=" * 70)
print("DISTRIBUIÇÃO DOS PATCHES POR TILE")
print("=" * 70)

print(f"\nNúmero de tiles: {len(tile_counts)}")
print(f"Número de patches: {len(df):,}")

print("\nPatches por tile:")
print(tile_counts.to_string())

print("\n" + "=" * 70)
print("TOTAL")
print("=" * 70)

print(f"Tiles:   {len(tile_counts)}")
print(f"Patches: {tile_counts.sum():,}")
print(f"Mínimo:  {tile_counts.min():,}")
print(f"Máximo:  {tile_counts.max():,}")
print(f"Média:   {tile_counts.mean():.2f}")
print(f"Mediana: {tile_counts.median():.2f}")

DISTRIBUIÇÃO DOS PATCHES POR TILE

Número de tiles: 115
Número de patches: 30,000

Patches por tile:
tile
S2B_MSIL2A_20180502T093039_N9999_R136_T34TEP    573
S2A_MSIL2A_20170704T112111_N9999_R037_T29SND    565
S2A_MSIL2A_20180506T100031_N9999_R122_T33UWP    541
S2B_MSIL2A_20180525T094029_N9999_R036_T35VNL    535
S2B_MSIL2A_20180515T094029_N9999_R036_T35VNJ    531
S2B_MSIL2A_20180515T112109_N9999_R037_T29SNC    526
S2B_MSIL2A_20170914T093029_N9999_R136_T34TEP    524
S2B_MSIL2A_20180525T094029_N9999_R036_T35VNK    516
S2A_MSIL2A_20171002T112111_N9999_R037_T29SNB    513
S2B_MSIL2A_20180421T114349_N9999_R123_T29UPU    511
S2A_MSIL2A_20170813T112121_N9999_R037_T29SNC    511
S2B_MSIL2A_20180515T112109_N9999_R037_T29SND    509
S2B_MSIL2A_20180326T112109_N9999_R037_T29SNB    508
S2B_MSIL2A_20170927T094019_N9999_R036_T35ULB    507
S2B_MSIL2A_20170924T093019_N9999_R136_T35VPK    504
S2A_MSIL2A_20171221T112501_N9999_R037_T29SND    501
S2A_MSIL2A_20171121T112351_N9999_R037_T29SND    501
S2A_MSIL2A

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

PROJECT_ROOT = Path(".")
CSV_PATH = PROJECT_ROOT / "data" / "selected_30k.csv"

OUTPUT_CSV = PROJECT_ROOT / "data" / "selected_30k_spatial.csv"
REPORT_PATH = PROJECT_ROOT / "data" / "spatial_split_report.txt"

SEED = 42

TARGETS = {
    "train": 0.70,
    "validation": 0.15,
    "test": 0.15,
}

# Número de tentativas da busca aleatorizada.
# 5.000 já é suficiente para encontrar soluções boas.
N_ITERATIONS = 5000


# ============================================================
# 2. CARREGAR DATASET
# ============================================================

df = pd.read_csv(CSV_PATH)

print("=" * 70)
print("DATASET ORIGINAL")
print("=" * 70)

print(f"Linhas: {len(df):,}")
print(f"Colunas: {len(df.columns)}")


# ============================================================
# 3. EXTRAIR TILE
# ============================================================

def get_tile_from_patch_id(patch_id):
    """
    Remove os dois últimos componentes do patch_id.

    Exemplo:
    S2B_..._T35VNL_00_28
    ->
    S2B_..._T35VNL
    """
    return patch_id.rsplit("_", 2)[0]


df["tile"] = df["patch_id"].apply(get_tile_from_patch_id)


# ============================================================
# 4. AUDITORIA INICIAL
# ============================================================

n_patches = len(df)
n_tiles = df["tile"].nunique()

tile_counts = (
    df.groupby("tile")
      .size()
      .sort_values(ascending=False)
)

print(f"Tiles:   {n_tiles}")
print(f"Patches: {n_patches:,}")

print("\nPatches por tile:")
print(tile_counts.describe())


# ============================================================
# 5. REPRESENTAÇÃO DOS TILES
# ============================================================

# Cada tile vira uma unidade indivisível.
#
# Além da quantidade de patches, guardamos as distribuições
# das variáveis categóricas para tentar equilibrar os splits.

categorical_columns = [
    "country",
    "season",
    "climate_zone",
]

tiles = (
    df.groupby("tile")
      .agg(
          n_patches=("patch_id", "size")
      )
      .reset_index()
)

print("\nNúmero de tiles:", len(tiles))


# ============================================================
# 6. CRIAR MATRIZES DE DISTRIBUIÇÃO
# ============================================================

# Para cada categoria, calculamos quantos patches pertencem
# a cada categoria dentro de cada tile.

feature_matrices = []

for column in categorical_columns:

    # Contagem tile x categoria
    matrix = pd.crosstab(
        df["tile"],
        df[column]
    )

    # Garante que a ordem dos tiles seja a mesma
    matrix = matrix.reindex(tiles["tile"]).fillna(0)

    # Normalizamos pela quantidade de patches do tile.
    # Isso representa a composição daquele tile.
    matrix = matrix.div(
        matrix.sum(axis=1).replace(0, 1),
        axis=0
    )

    feature_matrices.append(matrix.values)


# ============================================================
# 7. FUNÇÃO DE SCORE
# ============================================================

tile_names = tiles["tile"].values
patch_counts = tiles["n_patches"].values.astype(float)

n_tiles = len(tile_names)

split_names = ["train", "validation", "test"]

target_patch_counts = np.array([
    TARGETS["train"] * n_patches,
    TARGETS["validation"] * n_patches,
    TARGETS["test"] * n_patches,
])


def evaluate_assignment(assignment):
    """
    Avalia uma divisão dos tiles.

    assignment:
        array com valores:
        0 = train
        1 = validation
        2 = test

    Quanto menor o score, melhor.
    """

    score = 0.0

    # --------------------------------------------------------
    # A) Proporção de patches
    # --------------------------------------------------------

    actual_patch_counts = np.array([
        patch_counts[assignment == 0].sum(),
        patch_counts[assignment == 1].sum(),
        patch_counts[assignment == 2].sum(),
    ])

    # Erro relativo na quantidade de patches.
    patch_error = (
        np.abs(actual_patch_counts - target_patch_counts)
        / target_patch_counts
    )

    score += 1000 * patch_error.sum()


    # --------------------------------------------------------
    # B) Distribuição das variáveis categóricas
    # --------------------------------------------------------

    for matrix in feature_matrices:

        # Distribuição global
        global_distribution = (
            matrix * patch_counts[:, None]
        ).sum(axis=0)

        global_distribution = (
            global_distribution /
            global_distribution.sum()
        )

        for split_id in range(3):

            mask = assignment == split_id

            if not mask.any():
                score += 100000
                continue

            split_weight = patch_counts[mask]

            split_distribution = (
                matrix[mask] * split_weight[:, None]
            ).sum(axis=0)

            total = split_distribution.sum()

            if total == 0:
                continue

            split_distribution = split_distribution / total

            # Distância absoluta da distribuição global.
            distribution_error = np.abs(
                split_distribution - global_distribution
            ).sum()

            score += 50 * distribution_error


    # --------------------------------------------------------
    # C) Penalizar split vazio
    # --------------------------------------------------------

    for split_id in range(3):

        if not np.any(assignment == split_id):
            score += 100000


    return score


# ============================================================
# 8. GERAR UMA SOLUÇÃO INICIAL
# ============================================================

rng = np.random.default_rng(SEED)


def create_initial_assignment():

    # Começamos pelos maiores tiles.
    order = np.argsort(-patch_counts)

    assignment = np.full(n_tiles, -1)

    current_counts = np.zeros(3)

    # Distribui cada tile para o split que está mais distante
    # do seu alvo.
    for idx in order:

        deficits = target_patch_counts - current_counts

        # Entre os splits disponíveis, escolhemos o maior déficit.
        split = np.argmax(deficits)

        assignment[idx] = split
        current_counts[split] += patch_counts[idx]

    return assignment


best_assignment = create_initial_assignment()
best_score = evaluate_assignment(best_assignment)

print("\nScore inicial:", best_score)


# ============================================================
# 9. BUSCA LOCAL + RANDOMIZAÇÃO
# ============================================================

print("\nOtimizando divisão por tile...")

for iteration in range(N_ITERATIONS):

    # Começa da melhor solução encontrada.
    candidate = best_assignment.copy()

    # Faz entre 1 e 5 movimentos.
    n_moves = rng.integers(1, 6)

    for _ in range(n_moves):

        idx = rng.integers(0, n_tiles)

        old_split = candidate[idx]

        possible_splits = [0, 1, 2]
        possible_splits.remove(old_split)

        new_split = rng.choice(possible_splits)

        candidate[idx] = new_split

    candidate_score = evaluate_assignment(candidate)

    # Aceita somente melhorias.
    if candidate_score < best_score:

        best_assignment = candidate
        best_score = candidate_score

    # A cada certo número de iterações, também tentamos
    # trocar dois tiles entre splits.
    if iteration % 10 == 0:

        candidate = best_assignment.copy()

        idx1, idx2 = rng.choice(
            n_tiles,
            size=2,
            replace=False
        )

        candidate[idx1], candidate[idx2] = (
            candidate[idx2],
            candidate[idx1]
        )

        candidate_score = evaluate_assignment(candidate)

        if candidate_score < best_score:

            best_assignment = candidate
            best_score = candidate_score


print("Melhor score encontrado:", best_score)


# ============================================================
# 10. TRANSFORMAR ASSIGNMENT EM NOME DO SPLIT
# ============================================================

split_mapping = {
    0: "train",
    1: "validation",
    2: "test",
}

tiles["new_split"] = [
    split_mapping[x]
    for x in best_assignment
]


# ============================================================
# 11. APLICAR SPLIT AO DATASET
# ============================================================

tile_to_split = dict(
    zip(
        tiles["tile"],
        tiles["new_split"]
    )
)

df["split_original"] = df["split"]

df["split"] = df["tile"].map(tile_to_split)


# ============================================================
# 12. VERIFICAÇÕES FUNDAMENTAIS
# ============================================================

print("\n" + "=" * 70)
print("VALIDAÇÃO DO NOVO SPLIT")
print("=" * 70)


# Nenhum split nulo
assert df["split"].notna().all()


# Todos os patches continuam presentes
assert len(df) == n_patches


# Cada patch_id continua único
assert df["patch_id"].is_unique


# Cada tile pertence a apenas um split
tile_split_counts = (
    df.groupby("tile")["split"]
      .nunique()
)

assert tile_split_counts.max() == 1

print("✓ Todos os patches continuam presentes")
print("✓ patch_id continua único")
print("✓ Nenhum tile está dividido entre splits")


# ============================================================
# 13. DISTRIBUIÇÃO FINAL
# ============================================================

split_summary = (
    df.groupby("split")
      .agg(
          patches=("patch_id", "size"),
          tiles=("tile", "nunique")
      )
      .reindex(["train", "validation", "test"])
)

split_summary["proporcao"] = (
    split_summary["patches"] / n_patches
)

print("\nDistribuição final:")
print(split_summary)


# ============================================================
# 14. DISTRIBUIÇÃO DE COUNTRY
# ============================================================

country_distribution = pd.crosstab(
    df["country"],
    df["split"],
    normalize="columns"
) * 100

country_distribution = country_distribution[
    ["train", "validation", "test"]
]

print("\n" + "=" * 70)
print("COUNTRY (%)")
print("=" * 70)

print(country_distribution.round(2))


# ============================================================
# 15. DISTRIBUIÇÃO DE SEASON
# ============================================================

season_distribution = pd.crosstab(
    df["season"],
    df["split"],
    normalize="columns"
) * 100

season_distribution = season_distribution[
    ["train", "validation", "test"]
]

print("\n" + "=" * 70)
print("SEASON (%)")
print("=" * 70)

print(season_distribution.round(2))


# ============================================================
# 16. DISTRIBUIÇÃO DE CLIMATE ZONE
# ============================================================

climate_distribution = pd.crosstab(
    df["climate_zone"],
    df["split"],
    normalize="columns"
) * 100

climate_distribution = climate_distribution[
    ["train", "validation", "test"]
]

print("\n" + "=" * 70)
print("CLIMATE ZONE (%)")
print("=" * 70)

print(climate_distribution.round(2))


# ============================================================
# 17. VERIFICAR TILES POR SPLIT
# ============================================================

tiles_per_split = (
    tiles.groupby("new_split")
         .size()
         .reindex(["train", "validation", "test"])
)

print("\nTiles por split:")
print(tiles_per_split)


# ============================================================
# 18. SALVAR NOVO DATASET
# ============================================================

# Mantemos a coluna split_original para auditoria.
# Depois, se quiser, podemos removê-la.

df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n✓ Dataset salvo em:")
print(OUTPUT_CSV)


# ============================================================
# 19. GERAR RELATÓRIO
# ============================================================

report_lines = []

report_lines.append("=" * 70)
report_lines.append("RELATÓRIO DO SPLIT ESPACIAL")
report_lines.append("=" * 70)

report_lines.append("")
report_lines.append(f"Seed: {SEED}")
report_lines.append(f"Iterações: {N_ITERATIONS}")
report_lines.append(f"Total de patches: {n_patches:,}")
report_lines.append(f"Total de tiles: {n_tiles}")
report_lines.append("")

report_lines.append("DISTRIBUIÇÃO FINAL")
report_lines.append("-" * 70)
report_lines.append(
    split_summary.to_string()
)

report_lines.append("")
report_lines.append("TILES POR SPLIT")
report_lines.append("-" * 70)
report_lines.append(
    tiles_per_split.to_string()
)

report_lines.append("")
report_lines.append("COUNTRY (%)")
report_lines.append("-" * 70)
report_lines.append(
    country_distribution.round(2).to_string()
)

report_lines.append("")
report_lines.append("SEASON (%)")
report_lines.append("-" * 70)
report_lines.append(
    season_distribution.round(2).to_string()
)

report_lines.append("")
report_lines.append("CLIMATE ZONE (%)")
report_lines.append("-" * 70)
report_lines.append(
    climate_distribution.round(2).to_string()
)

report_lines.append("")
report_lines.append("VERIFICAÇÕES")
report_lines.append("-" * 70)
report_lines.append("✓ 30.000 patches preservados")
report_lines.append("✓ patch_id único")
report_lines.append("✓ nenhum tile compartilhado entre splits")
report_lines.append("✓ split sem valores nulos")


with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))


print("\n✓ Relatório salvo em:")
print(REPORT_PATH)


# ============================================================
# 20. MOSTRAR ALGUNS TILES DE CADA SPLIT
# ============================================================

print("\n" + "=" * 70)
print("EXEMPLO DE TILES POR SPLIT")
print("=" * 70)

for split in ["train", "validation", "test"]:

    print(f"\n[{split.upper()}]")

    subset = (
        tiles[tiles["new_split"] == split]
        .sort_values("n_patches", ascending=False)
        .head(10)
    )

    for _, row in subset.iterrows():

        print(
            f"{row['tile']} -> "
            f"{row['n_patches']} patches"
        )

DATASET ORIGINAL
Linhas: 30,000
Colunas: 9
Tiles:   115
Patches: 30,000

Patches por tile:
count    115.000000
mean     260.869565
std      173.139510
min        1.000000
25%      112.500000
50%      237.000000
75%      428.500000
max      573.000000
dtype: float64

Número de tiles: 115

Score inicial: 143.08237902872776

Otimizando divisão por tile...
Melhor score encontrado: 136.42435305060945

VALIDAÇÃO DO NOVO SPLIT
✓ Todos os patches continuam presentes
✓ patch_id continua único
✓ Nenhum tile está dividido entre splits

Distribuição final:
            patches  tiles  proporcao
split                                
train         20996     61   0.699867
validation     4500     28   0.150000
test           4504     26   0.150133

COUNTRY (%)
split        train  validation   test
country                              
Austria       7.58       16.78  12.94
Belgium       1.25        6.58   4.11
Finland      32.07       29.82  34.86
Ireland      10.33        4.98   9.92
Kosovo        0.53

In [ ]:
df_train = df[df["split"] == "train"].drop(columns=["split","split_original","output","latitude","longitude","country","season","climate_zone","tile"])
df_validation = df[df["split"] == "validation"].drop(columns=["split","split_original","output","latitude","longitude","country","season","climate_zone","tile"])
df_test = df[df["split"] == "test"].drop(columns=["split","split_original","output","latitude","longitude","country","season","climate_zone","tile"])
df_train.to_csv("data/train.csv", index=False)
df_validation.to_csv("data/validation.csv", index=False)
df_test.to_csv("data/test.csv", index=False)